# 퍼셉트론 (1958) — 어디까지 되고 어디서 멈추는가

Rosenblatt 의 학습 규칙을 그대로 구현해 **노트에 적어 둔 한계 네 가지가 실제로 그 모양으로 나타나는지**를 확인한다.

돌리는 것 다섯 가지다.

| 실험 | 묻는 것 |
|---|---|
| A | AND · OR · XOR 에서 규칙이 멈추는가 |
| B | 멈출 때까지의 틀린 횟수가 수렴 정리의 한계 (R/γ)² 안에 들어가는가 |
| C | 멈춘 경계는 여유(margin)가 가장 큰 경계인가 |
| D | 네 점에 줄 수 있는 16 가지 라벨링 중 직선 하나로 갈리는 것은 몇 개인가 |
| E | 학습률 η 가 결과를 바꾸는가 |

**돌리는 법.** 위에서 아래로 모두 실행한다. CPU 만 쓰고 바깥 데이터를 읽지 않는다.
결과는 `results/` 에 CSV 로, 그림은 `figures/` 에 PNG 로 떨어진다. 맨 아래 칸이 요약을 찍는다.

In [ ]:
# ── 준비 ────────────────────────────────────────────────────────────────
import os, sys, time, itertools, platform
import numpy as np
import pandas as pd

try:
    from scipy.optimize import linprog
    HAVE_SCIPY = True
except Exception as e:
    HAVE_SCIPY = False
    print("scipy 가 없다:", e)

NOTEBOOK = "perceptron_1958.ipynb"
RUN_ID   = time.strftime("%Y%m%d_%H%M%S")
HERE     = os.getcwd()
RESULTS  = os.path.join(HERE, "results")
FIGURES  = os.path.join(HERE, "figures")
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

ENV = {
    "run_id": RUN_ID, "notebook": NOTEBOOK,
    "python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
    "scipy": "없음" if not HAVE_SCIPY else __import__("scipy").__version__,
    "platform": platform.platform(), "cwd": HERE,
}
for k, v in ENV.items():
    print("%-10s %s" % (k, v))


def save(df, name):
    """결과 CSV 에 재현 정보를 열로 붙여 저장한다 (experiments/README.md 규칙 3)."""
    df = df.copy()
    for k in ("run_id", "notebook", "python", "numpy", "scipy", "platform"):
        df[k] = ENV[k]
    path = os.path.join(RESULTS, name)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("저장:", name, df.shape)
    return df

## 0. 규칙 구현

퍼셉트론은 **가중치를 더해 문턱을 넘으면 1, 아니면 0** 을 내고, **틀렸을 때만** 그 입력 방향으로 가중치를 옮긴다.

$$\hat{y} = \mathbb{1}[\,w \cdot x + b \ge 0\,], \qquad w \leftarrow w + \eta\,(y - \hat{y})\,x, \qquad b \leftarrow b + \eta\,(y - \hat{y})$$

문턱은 값이 늘 1 인 입력을 하나 더 두어 가중치의 하나로 다룬다.

In [ ]:
def perceptron(X, y, eta=1.0, max_sweeps=100000, w0=None, seed=0):
    """퍼셉트론 학습 규칙. 한 훑기 = 학습 예 전부를 한 번 (무작위 순서로) 지나는 것.

    돌려주는 것: (멈췄는가, 훑기 수, 가중치를 고친 횟수, 최종 가중치)
    마지막 성분이 문턱이다."""
    rng = np.random.default_rng(seed)
    n, d = X.shape
    Z = np.hstack([X, np.ones((n, 1))])          # 늘 1 인 입력을 붙인다
    w = np.zeros(d + 1) if w0 is None else np.array(w0, dtype=float)
    updates = 0
    for t in range(1, max_sweeps + 1):
        wrong = 0
        for i in rng.permutation(n):
            pred = 1.0 if Z[i] @ w >= 0 else 0.0
            if pred != y[i]:
                w = w + eta * (y[i] - pred) * Z[i]
                updates += 1
                wrong += 1
        if wrong == 0:
            return True, t, updates, w
    return False, max_sweeps, updates, w


def margin_of(X, y, w):
    """가중치 w 가 만든 경계와 가장 가까운 점 사이의 거리. 부호가 음수면 틀린 점이 있다는 뜻이다."""
    Z = np.hstack([X, np.ones((len(X), 1))])
    s = np.where(y >= 0.5, 1.0, -1.0)
    nrm = np.linalg.norm(w[:-1])
    if nrm == 0:
        return 0.0
    return float(np.min(s * (Z @ w)) / nrm)


def max_margin(X, y):
    """여유를 가장 크게 잡는 경계의 여유. 선형계획법으로 구한다. scipy 가 없으면 None."""
    if not HAVE_SCIPY:
        return None
    n, d = X.shape
    Z = np.hstack([X, np.ones((n, 1))])
    s = np.where(y >= 0.5, 1.0, -1.0)
    # |w| 를 1 로 고정하는 대신, s_i (Z_i . w) >= 1 을 만족하는 w 중 |w| 가 가장 작은 것을 찾는다.
    # 선형계획법으로는 |w| 를 직접 줄일 수 없으므로 방향을 격자로 훑어 가장 큰 여유를 고른다.
    best = -np.inf
    for th in np.linspace(0, np.pi, 721)[:-1] if d == 2 else []:
        u = np.array([np.cos(th), np.sin(th)])
        proj = X @ u
        lo = proj[y >= 0.5].min() if (y >= 0.5).any() else None
        hi = proj[y < 0.5].max() if (y < 0.5).any() else None
        if lo is None or hi is None:
            continue
        gap = lo - hi                     # 양수면 이 방향으로 갈린다
        gap2 = proj[y < 0.5].min() - proj[y >= 0.5].max()
        best = max(best, gap / 2.0, gap2 / 2.0)
    return float(best) if np.isfinite(best) else None


def separable(X, y):
    """직선(평면) 하나로 갈리는지를 선형계획법으로 판정한다."""
    if not HAVE_SCIPY:
        return None
    n, d = X.shape
    Z = np.hstack([X, np.ones((n, 1))])
    s = np.where(y >= 0.5, 1.0, -1.0)
    res = linprog(c=np.zeros(d + 1), A_ub=-(s[:, None] * Z), b_ub=-np.ones(n),
                  bounds=[(None, None)] * (d + 1), method="highs")
    return bool(res.status == 0)


# 손으로 한 번 확인한다 — AND 는 갈린다
X4 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_and = np.array([0, 0, 0, 1], dtype=float)
ok, sweeps, upd, w = perceptron(X4, y_and)
print("AND: 멈췄는가 %s, 훑기 %d, 가중치를 고친 횟수 %d, w = %s" % (ok, sweeps, upd, np.round(w, 2)))

## A. AND · OR · XOR — 멈추는가

**H1**: AND 와 OR 는 모든 seed 에서 멈춘다. **H2**: XOR 는 어떤 seed 에서도 멈추지 않는다.

틀리면 어떤 모양인가를 미리 적어 둔다 — H2 가 틀렸다면 XOR 에서 멈춘 seed 가 하나라도 나올 것이고,
그러면 구현이 규칙을 벗어난 것이다(예: 문턱 갱신의 부호).

In [ ]:
TASKS = {
    "AND": np.array([0, 0, 0, 1], dtype=float),
    "OR":  np.array([0, 1, 1, 1], dtype=float),
    "XOR": np.array([0, 1, 1, 0], dtype=float),
}
MAX_SWEEPS_A = 2000
SEEDS_A = 50

rows = []
t0 = time.time()
for name, y in TASKS.items():
    for s in range(SEEDS_A):
        ok, sw, upd, w = perceptron(X4, y, eta=1.0, max_sweeps=MAX_SWEEPS_A, seed=s)
        rows.append(dict(task=name, seed=s, converged=ok, sweeps=sw, updates=upd,
                         max_sweeps=MAX_SWEEPS_A, eta=1.0,
                         w1=w[0], w2=w[1], threshold=w[2],
                         margin=margin_of(X4, y, w) if ok else np.nan))
A = save(pd.DataFrame(rows), "A_tasks.csv")
print("%.1f초" % (time.time() - t0))

print()
print(A.groupby("task").agg(멈춘_seed=("converged", "sum"),
                            훑기_중앙값=("sweeps", "median"),
                            갱신_중앙값=("updates", "median"),
                            여유_중앙값=("margin", "median")).to_string())
print()
print("XOR 에서 멈춘 seed 수:", int(A[A.task == 'XOR'].converged.sum()),
      "/", SEEDS_A, " (훑기 상한 %d 안에서)" % MAX_SWEEPS_A)

## B. 수렴 정리의 한계 (R/γ)²

두 묶음이 여유 γ 로 갈리고 모든 입력의 길이가 R 이하이면 **틀리는 횟수가 (R/γ)² 를 넘지 않는다**는 것이 수렴 정리다.
여유를 세 가지로 두고 실제 갱신 횟수가 그 한계 안에 들어가는지 본다.

**H1**: 모든 행에서 `실제 갱신 ≤ (R/γ)²` 이다. **H1-a**: 한 행이라도 넘으면 구현이나 γ 계산이 틀린 것이다.

In [ ]:
def make_separable(gamma_target, n=60, seed=0, dim=2):
    """정답 경계에서 gamma_target 이상 떨어진 점만 모아 선형 분리 가능한 데이터를 만든다."""
    rng = np.random.default_rng(seed)
    wstar = np.ones(dim) / np.sqrt(dim)
    X = []
    y = []
    while len(X) < n:
        p = rng.uniform(-1, 1, dim)
        m = p @ wstar
        if abs(m) >= gamma_target:
            X.append(p)
            y.append(1.0 if m > 0 else 0.0)
    return np.array(X), np.array(y), wstar


rows = []
for gt in (0.05, 0.1, 0.2, 0.3):
    for s in range(20):
        X, y, wstar = make_separable(gt, n=60, seed=s)
        Z = np.hstack([X, np.ones((len(X), 1))])
        R = float(np.max(np.linalg.norm(Z, axis=1)))
        wn = np.append(wstar, 0.0)
        wn = wn / np.linalg.norm(wn)
        sgn = np.where(y >= 0.5, 1.0, -1.0)
        gamma = float(np.min(sgn * (Z @ wn)))
        ok, sw, upd, w = perceptron(X, y, max_sweeps=100000, seed=s)
        rows.append(dict(gamma_target=gt, seed=s, n=len(X), R=R, gamma=gamma,
                         bound=(R / gamma) ** 2, converged=ok, sweeps=sw, updates=upd,
                         within_bound=bool(upd <= (R / gamma) ** 2)))
B = save(pd.DataFrame(rows), "B_bound.csv")

print()
print(B.groupby("gamma_target").agg(gamma_중앙값=("gamma", "median"),
                                    R_중앙값=("R", "median"),
                                    한계_중앙값=("bound", "median"),
                                    실제갱신_중앙값=("updates", "median"),
                                    실제갱신_최대=("updates", "max"),
                                    한계_안=("within_bound", "sum")).to_string())
print()
print("한계를 넘은 행:", int((~B.within_bound).sum()), "/", len(B))

## C. 멈춘 경계가 가장 좋은 경계는 아니다

수렴 정리가 보장하는 것은 **갈라 놓는 어떤 선**이지 **여유가 가장 큰 선**이 아니다.
멈췄을 때의 여유를, 같은 데이터에서 얻을 수 있는 가장 큰 여유와 견준다.

**H1**: 퍼셉트론이 멈춘 경계의 여유가 최대 여유보다 작은 행이 대부분이다.
**H1-a**: 둘이 늘 같다면 규칙이 여유를 키우는 성질을 가진 것이고, 그러면 이 한계를 다시 적어야 한다.

In [ ]:
rows = []
for gt in (0.05, 0.1, 0.2):
    for s in range(20):
        X, y, _ = make_separable(gt, n=40, seed=100 + s)
        ok, sw, upd, w = perceptron(X, y, max_sweeps=100000, seed=s)
        if not ok:
            rows.append(dict(gamma_target=gt, seed=s, converged=False,
                             got=np.nan, best=np.nan, ratio=np.nan))
            continue
        got = margin_of(X, y, w)
        best = max_margin(X, y)
        rows.append(dict(gamma_target=gt, seed=s, converged=True, got=got, best=best,
                         ratio=(got / best) if best else np.nan))
C = save(pd.DataFrame(rows), "C_margin.csv")

print()
ok_rows = C[C.converged]
print("멈춘 행", len(ok_rows), "/", len(C))
if len(ok_rows) and ok_rows.best.notna().any():
    print("얻은 여유 / 가장 큰 여유 — 중앙값 %.3f, 최소 %.3f, 최대 %.3f"
          % (ok_rows.ratio.median(), ok_rows.ratio.min(), ok_rows.ratio.max()))
    print("얻은 여유가 가장 큰 여유보다 작은 행:", int((ok_rows.ratio < 0.999).sum()), "/", len(ok_rows))
else:
    print("scipy 가 없어 가장 큰 여유를 구하지 못했다 — 이 실험은 이번 실행에서 시험하지 못했다.")

## D. 네 점의 라벨링 16 가지 중 몇 개가 직선 하나로 갈리는가

XOR 는 **가장 작은 반례 하나**일 뿐이다. 네 점에 줄 수 있는 모든 라벨링을 세어 보면
퍼셉트론이 표현할 수 있는 것의 범위가 한눈에 보인다.

**H1**: 16 가지 중 14 가지가 갈리고, 갈리지 않는 둘은 XOR 와 그 반대(XNOR)다.

In [ ]:
rows = []
for bits in itertools.product([0, 1], repeat=4):
    y = np.array(bits, dtype=float)
    sep = separable(X4, y)
    ok, sw, upd, w = perceptron(X4, y, max_sweeps=2000, seed=0)
    rows.append(dict(labeling="".join(str(b) for b in bits),
                     lp_separable=sep, perceptron_stopped=ok, sweeps=sw, updates=upd))
D = save(pd.DataFrame(rows), "D_dichotomies.csv")

n_sep = int(D.lp_separable.sum()) if D.lp_separable.notna().all() else None
print()
print("선형계획법으로 갈린다고 나온 라벨링:", n_sep, "/ 16")
print("퍼셉트론이 2000 훑기 안에 멈춘 라벨링:", int(D.perceptron_stopped.sum()), "/ 16")
print()
print("멈추지 않은 것:")
print(D[~D.perceptron_stopped][["labeling", "lp_separable", "updates"]].to_string(index=False))
print()
print("(0110 이 XOR, 1001 이 XNOR 다. 네 점의 차례는 (0,0) (0,1) (1,0) (1,1) 이다.)")

## E. 학습률 η 는 결과를 바꾸는가

**가중치를 0 에서 시작하면 η 는 전체 크기만 바꾸고 부호는 바꾸지 않는다.** 그러면 판단이 같아지므로
갱신 횟수도 같아야 한다. 0 이 아닌 곳에서 시작하면 달라진다.

**H1**: `w0 = 0` 이면 η 를 바꿔도 갱신 횟수가 같다. **H2**: `w0 ≠ 0` 이면 달라지는 행이 생긴다.

In [ ]:
rows = []
for gt in (0.1,):
    for s in range(15):
        X, y, _ = make_separable(gt, n=40, seed=200 + s)
        rng = np.random.default_rng(1000 + s)
        w_nonzero = rng.uniform(-1, 1, X.shape[1] + 1)
        for eta in (0.01, 0.1, 1.0, 10.0):
            for start, w0 in (("영에서", None), ("난수에서", w_nonzero)):
                ok, sw, upd, w = perceptron(X, y, eta=eta, max_sweeps=100000, w0=w0, seed=s)
                rows.append(dict(seed=s, eta=eta, start=start, converged=ok,
                                 sweeps=sw, updates=upd))
E = save(pd.DataFrame(rows), "E_eta.csv")

print()
piv = E.pivot_table(index=["start", "seed"], columns="eta", values="updates")
same = piv.loc["영에서"].nunique(axis=1).eq(1)
same2 = piv.loc["난수에서"].nunique(axis=1).eq(1)
print("영에서 시작 — η 넷의 갱신 횟수가 모두 같은 seed:", int(same.sum()), "/", len(same))
print("난수에서 시작 — η 넷의 갱신 횟수가 모두 같은 seed:", int(same2.sum()), "/", len(same2))
print()
print(piv.head(8).to_string())

## 그림

한글 글꼴이 없는 환경에서 네모로 깨지는 것을 피하려고 **축 라벨은 영문으로 둔다.**
그림의 뜻은 `README.md` 에 우리말로 적는다.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = []

# 1) 세 과제의 훑기 수
fig, ax = plt.subplots(figsize=(6.4, 3.6), dpi=140)
for i, (name, g) in enumerate(A.groupby("task")):
    ax.scatter(np.full(len(g), i) + np.random.uniform(-.12, .12, len(g)), g.sweeps,
               s=14, alpha=.6, label=name)
ax.set_xticks(range(len(TASKS)))
ax.set_xticklabels(list(A.groupby("task").groups.keys()))
ax.set_yscale("log")
ax.set_ylabel("sweeps until no mistake (log)")
ax.set_title("AND / OR stop, XOR runs to the cap")
ax.axhline(MAX_SWEEPS_A, color="crimson", ls="--", lw=1, label="cap")
ax.legend(fontsize=8)
fig.tight_layout()
p = os.path.join(FIGURES, "A_tasks.png"); fig.savefig(p); FIG.append(p); plt.close(fig)

# 2) 수렴 정리의 한계와 실제
fig, ax = plt.subplots(figsize=(5.2, 4.4), dpi=140)
ax.scatter(B.bound, B.updates, s=16, alpha=.7)
lim = [1, max(B.bound.max(), B.updates.max()) * 1.2]
ax.plot(lim, lim, color="crimson", lw=1, ls="--", label="y = x (the bound)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("bound  (R/gamma)^2"); ax.set_ylabel("actual number of updates")
ax.set_title("every run stays under the bound")
ax.legend(fontsize=8)
fig.tight_layout()
p = os.path.join(FIGURES, "B_bound.png"); fig.savefig(p); FIG.append(p); plt.close(fig)

# 3) 얻은 여유와 가장 큰 여유
ok_rows = C[C.converged & C.best.notna()]
if len(ok_rows):
    fig, ax = plt.subplots(figsize=(5.2, 4.4), dpi=140)
    ax.scatter(ok_rows.best, ok_rows.got, s=18, alpha=.7)
    lim = [0, ok_rows.best.max() * 1.1]
    ax.plot(lim, lim, color="crimson", lw=1, ls="--", label="equal")
    ax.set_xlabel("largest possible margin"); ax.set_ylabel("margin the rule stopped at")
    ax.set_title("stopping is not the same as the best boundary")
    ax.legend(fontsize=8)
    fig.tight_layout()
    p = os.path.join(FIGURES, "C_margin.png"); fig.savefig(p); FIG.append(p); plt.close(fig)

print("그림", len(FIG), "장")
for p in FIG:
    print("  ", os.path.basename(p))

## 요약 — 아래 출력을 그대로 `README.md` 의 결과 칸에 옮긴다

In [ ]:
print("=" * 78)
print("퍼셉트론 (1958) 실험 요약   run_id =", RUN_ID)
print("=" * 78)
print()
print("[A] 세 과제, seed %d 개, 훑기 상한 %d" % (SEEDS_A, MAX_SWEEPS_A))
for name, g in A.groupby("task"):
    print("    %-4s 멈춘 seed %2d/%2d   훑기 중앙값 %6.0f   갱신 중앙값 %6.0f"
          % (name, g.converged.sum(), len(g), g.sweeps.median(), g.updates.median()))
print()
print("[B] 수렴 정리, %d 행" % len(B))
print("    한계를 넘은 행 %d 개" % int((~B.within_bound).sum()))
for gt, g in B.groupby("gamma_target"):
    print("    여유 목표 %.2f -> gamma %.3f, 한계 %8.0f, 실제 갱신 중앙값 %5.0f (최대 %5.0f)"
          % (gt, g.gamma.median(), g.bound.median(), g.updates.median(), g.updates.max()))
print()
ok_rows = C[C.converged & C.best.notna()]
print("[C] 여유, 멈춘 행 %d 개" % len(ok_rows))
if len(ok_rows):
    print("    얻은 여유 / 가장 큰 여유  중앙값 %.3f  최소 %.3f  최대 %.3f"
          % (ok_rows.ratio.median(), ok_rows.ratio.min(), ok_rows.ratio.max()))
    print("    가장 큰 여유보다 작았던 행 %d / %d" % (int((ok_rows.ratio < 0.999).sum()), len(ok_rows)))
else:
    print("    이번 실행에서 시험하지 못했다 (scipy 없음)")
print()
print("[D] 네 점의 라벨링 16 가지")
print("    선형계획법으로 갈린다 %s / 16,  퍼셉트론이 멈췄다 %d / 16"
      % (int(D.lp_separable.sum()) if D.lp_separable.notna().all() else "확인 못 함",
         int(D.perceptron_stopped.sum())))
print("    멈추지 않은 라벨링: %s" % ", ".join(D[~D.perceptron_stopped].labeling))
print()
print("[E] 학습률")
piv = E.pivot_table(index=["start", "seed"], columns="eta", values="updates")
print("    영에서 시작 — η 넷의 갱신 횟수가 같은 seed %d / %d"
      % (int(piv.loc["영에서"].nunique(axis=1).eq(1).sum()), len(piv.loc["영에서"])))
print("    난수에서 시작 — η 넷의 갱신 횟수가 같은 seed %d / %d"
      % (int(piv.loc["난수에서"].nunique(axis=1).eq(1).sum()), len(piv.loc["난수에서"])))
print()
print("결과 파일")
for f in sorted(os.listdir(RESULTS)):
    print("   results/%s" % f)
for f in sorted(os.listdir(FIGURES)):
    print("   figures/%s" % f)
print("=" * 78)